**Run note:** execute this notebook's first setup/code cell before any later cells. Each notebook is designed to run independently and re-detect the dataset path on its own.

# 11 â€” Counterfactuals & Robustness Testing

**Counterfactuals**: Minimal edits that flip the model's prediction.
- Text side: token masking + RoBERTa MLM substitution
- Image side: progressive GradCAM patch occlusion

**Robustness testing**: How sensitive is the model to:
- OCR noise (character substitutions)
- Paraphrasing
- Low-resolution images

In [ ]:
import os
import json
import re
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from transformers import CLIPModel, CLIPProcessor, RobertaTokenizer, RobertaForMaskedLM

ON_KAGGLE = Path("/kaggle/input").is_dir()
JSONL_CANDIDATES = {
    "train": ["train.jsonl"],
    "dev": ["dev.jsonl", "dev_seen.jsonl", "dev_unseen.jsonl"],
    "test": ["test.jsonl", "test_seen.jsonl", "test_unseen.jsonl"],
}
IMAGE_DIR_CANDIDATES = ("img", "images")


def _has_image_dir(path: Path) -> bool:
    return any((path / name).is_dir() for name in IMAGE_DIR_CANDIDATES)


def _has_any_jsonl(path: Path, names) -> bool:
    return any((path / name).is_file() for name in names)


def _looks_like_dataset_root(path: Path) -> bool:
    return path.is_dir() and _has_image_dir(path) and _has_any_jsonl(path, JSONL_CANDIDATES["train"])


def detect_data_dir():
    for env_name in ("KAGGLE_DATA_DIR", "META_HATEFUL_MEME_DATA_DIR"):
        env_dir = os.environ.get(env_name, "").strip()
        if env_dir and _looks_like_dataset_root(Path(env_dir)):
            return Path(env_dir), f"env:{env_name}"

    kaggle_input = Path("/kaggle/input")
    default_candidate = kaggle_input / "meta-hateful-meme-detection" / "data"
    if _looks_like_dataset_root(default_candidate):
        return default_candidate, "default:/kaggle/input/meta-hateful-meme-detection/data"

    if ON_KAGGLE:
        for train_jsonl in sorted(kaggle_input.rglob("train.jsonl")):
            candidate = train_jsonl.parent
            if _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

        for candidate in sorted(kaggle_input.rglob("*")):
            if candidate.is_dir() and _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

    for candidate in (Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd(), Path.cwd().parent):
        if _looks_like_dataset_root(candidate):
            return candidate, f"local:{candidate}"

    return None, "not-found"


def resolve_split(base_dir, names):
    base_dir = Path(base_dir)
    for name in names:
        path = base_dir / name
        if path.is_file():
            return path
    for name in names:
        matches = sorted(base_dir.rglob(name))
        if matches:
            return matches[0]
    return None


DATA_DIR, data_source = detect_data_dir()
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Set KAGGLE_DATA_DIR or META_HATEFUL_MEME_DATA_DIR to the folder containing train.jsonl and img/."
    )

IMG_DIR = next((DATA_DIR / name for name in IMAGE_DIR_CANDIDATES if (DATA_DIR / name).is_dir()), None)
DEV_PATH = resolve_split(DATA_DIR, JSONL_CANDIDATES["dev"])
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEV_PATH is None:
    raise FileNotFoundError(f"Expected dev split under {DATA_DIR}")

DATA_DIR = str(DATA_DIR)
IMG_DIR = str(IMG_DIR) if IMG_DIR is not None else None
DEV_PATH = str(DEV_PATH)
OUTPUT_DIR = str(OUTPUT_DIR)

print(f"Using DATA_DIR : {DATA_DIR}")
print(f"Using IMG_DIR  : {IMG_DIR}")
print(f"Using source   : {data_source}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Device         : {DEVICE}")


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(l) for l in f])


dev_df = load_jsonl(DEV_PATH)
print(f"Dev: {len(dev_df)} | Device: {DEVICE}")

In [ ]:
from pathlib import Path


def resolve_image_path(data_dir, image_ref):
    data_dir = Path(data_dir)
    image_ref = Path(str(image_ref))

    candidates = []
    if image_ref.is_absolute():
        candidates.append(image_ref)

    candidates.extend([
        data_dir / image_ref,
        data_dir.parent / image_ref,
    ])

    if image_ref.parts:
        if image_ref.parts[0] in {"img", "images"} and len(image_ref.parts) > 1:
            stripped = Path(*image_ref.parts[1:])
            candidates.extend([
                data_dir / stripped,
                data_dir.parent / stripped,
            ])
        elif image_ref.parts[0] not in {"img", "images"}:
            candidates.extend([
                data_dir / "img" / image_ref,
                data_dir / "images" / image_ref,
                data_dir.parent / "img" / image_ref,
                data_dir.parent / "images" / image_ref,
            ])

    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find image '{image_ref}' relative to {data_dir}")

In [ ]:
# â”€â”€ Load main model (inline definitions) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class CLIPEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters(): p.requires_grad_(False)
    def forward(self, pv, ids, mask):
        return (F.normalize(self.clip.get_image_features(pixel_values=pv), -1),
                F.normalize(self.clip.get_text_features(input_ids=ids, attention_mask=mask), -1))

class CrossAttentionFusion(nn.Module):
    def __init__(self, d=512, h=4):
        super().__init__()
        self.i2t=nn.MultiheadAttention(d,h,batch_first=True); self.t2i=nn.MultiheadAttention(d,h,batch_first=True)
        self.ni=nn.LayerNorm(d); self.nt=nn.LayerNorm(d)
    def forward(self, i, t):
        is_=i.unsqueeze(1); ts=t.unsqueeze(1)
        ic,ia=self.i2t(is_,ts,ts); tc,ta=self.t2i(ts,is_,is_)
        return torch.cat([self.ni(i+ic.squeeze(1)),self.nt(t+tc.squeeze(1))],-1),ia,ta

class HatefulMemeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=CLIPEncoder(); self.fusion=CrossAttentionFusion()
        self.head=nn.Sequential(nn.Linear(1024,256),nn.GELU(),nn.Dropout(0.3),
                                  nn.Linear(256,128),nn.GELU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(self, pv, ids, mask):
        i,t=self.encoder(pv,ids,mask); f,ia,ta=self.fusion(i,t); return self.head(f),ia,ta

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model     = HatefulMemeClassifier().to(DEVICE)
ckpt      = os.path.join(OUTPUT_DIR, "cross_attention_best.pt")
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); print("Checkpoint loaded.")
else:
    print("WARNING: checkpoint not found â€” using random weights.")
model.eval();

In [ ]:
    with torch.no_grad():
        logits, _, _ = model(enc["pixel_values"].to(DEVICE),
                              enc["input_ids"].to(DEVICE),
                              enc["attention_mask"].to(DEVICE))
    return torch.softmax(logits, -1)[0, 1].item()

# Test on one sample
row = dev_df[dev_df["label"] == 1].iloc[0]
img = Image.open(resolve_image_path(DATA_DIR, row["img"])).convert("RGB")
p   = predict(img, row["text"])
print(f"Sample text: '{row.text}'")
print(f"P(hateful)  = {p:.4f}")

In [ ]:
# â”€â”€ 11.1 Text counterfactuals via RoBERTa MLM â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rob_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
rob_mlm       = RobertaForMaskedLM.from_pretrained("roberta-base").to(DEVICE)
rob_mlm.eval();

def text_counterfactual(pil_img, original_text, n_candidates=5, threshold_flip=0.3):
    """
    For each token in the text, mask it and use RoBERTa to propose substitutes.
    Accept the first substitute that flips the prediction below threshold_flip.
    """
    orig_prob = predict(pil_img, original_text)
    tokens    = original_text.split()
    results   = []

    for i, token in enumerate(tokens):
        masked_text = " ".join(tokens[:i] + ["<mask>"] + tokens[i+1:])
        inputs = rob_tokenizer(masked_text, return_tensors="pt", truncation=True, max_length=128)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            out = rob_mlm(**inputs)

        mask_idx   = (inputs["input_ids"][0] == rob_tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        if len(mask_idx) == 0: continue
        top_ids    = out.logits[0, mask_idx[0]].topk(n_candidates).indices
        candidates = rob_tokenizer.convert_ids_to_tokens(top_ids)

        for cand in candidates:
            if cand == token: continue
            new_text = " ".join(tokens[:i] + [cand] + tokens[i+1:])
            new_prob = predict(pil_img, new_text)
            if new_prob < threshold_flip and orig_prob >= threshold_flip:
                results.append({
                    "original": original_text, "counterfactual": new_text,
                    "changed_token": token, "new_token": cand,
                    "orig_prob": orig_prob, "cf_prob": new_prob
                })
                return results  # return on first successful flip

    return results  # no flip found

# Test on hateful sample
cf_results = text_counterfactual(img, row["text"])
if cf_results:
    r = cf_results[0]
    print(f"Original      : '{r['original']}'  P={r['orig_prob']:.3f}")
    print(f"Counterfactual: '{r['counterfactual']}'  P={r['cf_prob']:.3f}")
    print(f"Changed token : '{r['changed_token']}' â†’ '{r['new_token']}'")
else:
    print("No flip found in this sample. Try increasing n_candidates.")

In [ ]:
# â”€â”€ 11.2 Image counterfactuals via patch occlusion â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def image_counterfactual(pil_img, text, patch_size=32, threshold_flip=0.3):
    """
    Progressive GradCAM-guided patch blacking-out.
    Occludes the most important image regions until prediction flips.
    """
    w, h       = pil_img.size
    orig_prob  = predict(pil_img, text)
    img_arr    = np.array(pil_img).copy()

    patches_tried = 0
    for row_start in range(0, h, patch_size):
        for col_start in range(0, w, patch_size):
            candidate_arr = img_arr.copy()
            row_end  = min(row_start + patch_size, h)
            col_end  = min(col_start + patch_size, w)
            candidate_arr[row_start:row_end, col_start:col_end] = 128  # grey out

            candidate_img = Image.fromarray(candidate_arr)
            new_prob      = predict(candidate_img, text)
            patches_tried += 1

            if new_prob < threshold_flip and orig_prob >= threshold_flip:
                return candidate_img, (row_start, col_start, row_end, col_end), orig_prob, new_prob, patches_tried
            img_arr = candidate_arr  # keep masking

    return Image.fromarray(img_arr), None, orig_prob, predict(Image.fromarray(img_arr), text), patches_tried

cf_img, region, op, np_, n_patches = image_counterfactual(img, row["text"])
print(f"Image patch occlusion: tried {n_patches} patches")
print(f"Original P(hate)={op:.3f} â†’ After P(hate)={np_:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img);    axes[0].set_title(f"Original P={op:.3f}");   axes[0].axis("off")
axes[1].imshow(cf_img); axes[1].set_title(f"After occlusion P={np_:.3f}"); axes[1].axis("off")
plt.suptitle("Image Counterfactual â€” Patch Occlusion", fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "image_counterfactual.png"), dpi=150)
plt.show()

In [ ]:
#  11.3 Robustness: OCR noise injection 

def inject_ocr_noise(text, noise_rate=0.1, seed=42):
    """Simulate OCR errors: random character substitutions."""
    rng   = random.Random(seed)
    chars = list(text)
    noise_map = {"a":"@", "e":"3", "i":"1", "o":"0", "s":"$", "g":"9", "l":"|", "t":"+"}
    for i, c in enumerate(chars):
        if rng.random() < noise_rate and c.lower() in noise_map:
            chars[i] = noise_map[c.lower()]
    return "".join(chars)

# Test on a batch of 20 samples
sample_rows = dev_df.sample(20, random_state=42)
orig_probs, noisy_probs = [], []

for _, row in sample_rows.iterrows():
    try:
        pil = Image.open(resolve_image_path(DATA_DIR, row["img"])).convert("RGB")
        op  = predict(pil, row["text"])
        np_ = predict(pil, inject_ocr_noise(row["text"]))
        orig_probs.append(op); noisy_probs.append(np_)
    except: pass

orig_probs  = np.array(orig_probs)
noisy_probs = np.array(noisy_probs)
prob_shift  = np.abs(orig_probs - noisy_probs)

print(f"OCR Noise Robustness (10% char substitution):")
print(f"  Avg probability shift : {prob_shift.mean():.4f}")
print(f"  Max probability shift : {prob_shift.max():.4f}")
print(f"  Prediction flips (>0.5 threshold): {((orig_probs>=0.5) != (noisy_probs>=0.5)).sum()}")

plt.figure(figsize=(6, 4))
plt.scatter(orig_probs, noisy_probs, alpha=0.7, c="#2196F3", edgecolor="white")
plt.plot([0,1],[0,1],"k--",alpha=0.5)
plt.xlabel("Original P(hate)"); plt.ylabel("Noisy OCR P(hate)")
plt.title("Robustness: OCR Noise", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
#  11.4 Robustness: image resolution degradation 
resolutions = [224, 112, 56, 28]
res_probs   = {}

test_row = dev_df[dev_df["label"] == 1].iloc[0]
orig_img = Image.open(resolve_image_path(DATA_DIR, test_row["img"])).convert("RGB")

for res in resolutions:
    degraded = orig_img.resize((res, res), Image.BILINEAR).resize((224, 224), Image.NEAREST)
    p = predict(degraded, test_row["text"])
    res_probs[res] = p
    print(f"  Resolution {res}x{res}    P(hate)={p:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(res_probs.keys()), list(res_probs.values()), marker="o", color="#9C27B0")
axes[0].invert_xaxis()
axes[0].set_xlabel("Resolution")
axes[0].set_ylabel("P(hate)")
axes[0].set_title("Effect of Downscaling")
axes[1].imshow(orig_img)
axes[1].set_title(f"Original image\nP={res_probs[224]:.3f}")
axes[1].axis("off")
plt.suptitle("Robustness: Resolution Degradation", fontweight="bold")
plt.tight_layout(); plt.show()